# H2Omics Google Colab: Part II

# Background

This notebook is designed to provide an accesible platform for analysing sequencing data. It allows you to perfom microbiome analysis without the need to dowload or install any additional software on your computer. The only requirement is a Google account and raw Illumina sequence data in the right format.

This notebook is designed to process and visualize  `.qza` files (QIIME2 output). If you do not yet have these  files supplied by your sequencing service company, you will need to generate them first by performing a QIIME2 analysis for Illumina sequencing data: [Google_Colab_H2Omics_PartI](https://colab.research.google.com/github/stijnteunissen/Workshop_H2Omics_test/blob/main/Google_Colab_H2Omics_PartI.ipynb?copy=true) notebook.

If you would like to use this notebook with Nanopore data instead, you can try this link [here](https://github.com/timyerg/NaMeco) to convert your Nanopore data into `.qza` files so it can be used with this notebook. Please note that we have not tested this process.

**Note:** Sometimes Google Colab reports an error saying that a package is missing or that something is wrong with the environment. We suggest in such cases that you restart the environment, by executing every code block from the very start of this Part II document. The easiest way to do this is to go to **Runtime > Disconnect and delete runtime** in the menu bar at the top of the notebook. Once the runtime restarted successfully, `play` the code blocks one-by-one.

# 1.0 Connecting to Google Drive

Before you can start uploading and analyzing microbiome data, you need to connect this notebook document to your Google Drive. This step is crucial because it allows you to save output files directly to Google Drive. Without this step, any files created during this Colab session will be lost once the Colab environment is closed (or when connection is interrupted).

The next section has a different look. It is code in a so-called code block. It is advised not to change text in the code blocks. The code can be executed by clicking the `play` button in the top left corner. In this notebook, each code block will be introduced with a brief statement describing what analysis step will be performed by the code.

For example, by the code block below you will install the `rpy2 library`, which is software to run R scripts directly within the Python environment of this notebook. This enables an integration of R statistical and microbiome analysis functions required for obtaining interpretable insights into your microbiome data.

# **Warning: Do not modify any of the code blocks!!**
Altering the code may break the pipeline and lead to unexpected errors in the analysis. Only run code blocks using the topleft `play` button.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rpy2
%load_ext rpy2.ipython
!git clone https://github.com/stijnteunissen/Workshop_H2Omics_test.git

In [ ]:
%%bash
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
mkdir -p /content/envs
./bin/micromamba create -q -y -p /content/envs/epa-env -c bioconda -c conda-forge epa-ng hmmer

# 2.0 Analysis Preparation

## 2.1 Creating the Output Folder

This code block creates a directory in your Google Drive within the home folder ("drive/MyDrive/"). The new directory will be created and named after choosing a self-defined project name. All output files generated during the analysis will be stored in this directory, ensuring logical organization and storage in your Google Drive. After running the code, write a project name (prevent use of spaces by using underscores "_"), and press confirm to set the project name.

If you have your own .qza files already, or have run the QIIME2 analysis elsewhere, you can start a new project here. In that case, select “New Project” and proceed as usual. However, if you previously ran the QIIME2 notebook Part I, and created a project there, choose that exact project from the dropdown menu.

In [ ]:
import sys
sys.path.append('/content/Workshop_H2Omics_test/H2Omics_analysis')

from starting_project import starting_project

starting_project()

## 2.2 Loading Input Files

In the next code block, you will upload the required microbiome data files from your local computer. If you have already run the Part I QIIME2 analysis notebook, you are only required to upload additional sample **metadata_extra** file. However, if you intend using `.qza` files not created by the Part I notebook, you must upload the required files listed below to proceed with the analysis. These files are essential for the remainder of the workflow.

The files include:

- **Feature Table:** This data matrix contains the sequence read counts of all microbial features (e.g., OTUs or ASVs) for all the samples in sequencing project. **(table.qza)**
- **Taxonomy Table:** This table contains the hierarchical taxonomic information (Kingdom, Phylum, Class, Order, Family, Genus, Species) that each microbial feature is classified to. **(classifier.qza)**
- **Phylogenetic Tree:** The phylogeny represents the evolutionary relationships among the microbial features based on their DNA sequence similarity. **(rooted-tree.qza)**
- **Representative Sequences:** This artifact contains the nucleotide sequences for each microbial feature (e.g., OTUs or ASVs). **(representative_sequences.qza)**
- **Sample Metadata:** This table contains contextual infromation for each sample in you sequencing project, including uniques sample identifiers and experimental or technical details (e.g., SampleID, BarcodeSequence, LinkerPrimerSequence, BarcodeName, ReversePrimer, etc.). **(metadata)**
- **Sample Metadata Extra:** This table includes information describing the project's samples in detail, including sampling date, experimental factors and additional measurements (e.g., pH, DNA concentration, chemical data, etc.). The following columns must always be present:
  - **sample_or_control**: Specifies whether a sample is a *sample*, *mock*, or *blank*.
  - **DNA_Concentration**: Reports the DNA concentration of the sequenced sample in ng/µl.
  - **na_type**: Indicates whether the sample is *DNA* or *RNA*.
  
  The **metadata_extra** file must be created by you before proceeding. Below is a linked template (with an example) to help you structure this file:
  [Template for metadata_extra](https://raw.githubusercontent.com/stijnteunissen/Workshop_H2Omics_test/refs/heads/main/H2Omics_analysis/Template_metadata_extra.xlsx)
  
  In this template, you can see an example of the columns that need to be added in the metadata extra file, along with an example of the experimental data, which in this example is *timepoint*. You can add any additional experimental details (columns) to the metadata file. Use "NA" to fill any missing values, and replace spaces or special charaters in column names with underscores. **(metadata_extra)**

Providing these files allows for their integrative analysis of your sequencing data.
The code block below also enables OPTIONAL addition of extra data on the samples. If not already included in the Sample Metadata Extra, the code block below helps adding extra data to the Sample Metadata. In the context of microbiome analysis, a specific example is biomass data to integrate information on 'microbial load'. Two explicit options typically applied in the water context are made possible in the notebook:
- **Flow cytometry (FCM)** based cell concentration data.
  - This table contains **SampleID** and the corresponding **cells_per_ml** value, representing the number of cells per milliliter sample. **(fcm)**
- **qPCR** based 16S rRNA gene copy concentration data.
  - This table contains a **SampleID** column and the gene copy concentration (**sq_calc_mean**). Note that this needs to be normalized to copies per ml sample.

The FCM and qPCR data can be added to the metadata_extra file or uploaded as separate files, and the script will combine these files for you.

**NOTES:**
* In additional data files, Sample IDs should match identically with Sample Metadata.
* The column names shown in bold, which are required, must match exatcly as indicated.
* QIIME2 output files must always use the .qza extension. **metadata** and **metadata_extra** files and optional qPCR or FCM data files must be in one of these extension formats: .txt, .tsv, or .csv.
* Additionally, make sure each filename matches the required name shown in parentheses for its file type.

In [ ]:
import sys
sys.path.append('/content/Workshop_H2Omics_test/H2Omics_analysis')

from import_files import import_files

import_files()

## 2.3 Parameter Selection

In the next code block, you will define a set of parameters required to run the analysis. You can set them all at once in the code block below.

* **Factors/covariates:** First, specify which sample metadata factors you want to examine, for example, *treatment* group (treated vs. untreated), sampling date, temperature, etc. Enter the exact column names you used in you (metadata_extra.tsv).
* **normalisation method:** choose the method used to quantify biomass:
  * `qPCR`
  * `FCM` (flow cytometry)
  * `NULL` (if no biomass measuremnet is available)
* **Blank and Mock:** indicate whether you dataset contains a blank (negative control) and mock (positive control) sample(s):
 * `TRUE` if present
 * `FALSE` if absent
* **Beta diversity aesthetics:** select which factor/covariate to use for plotting beta diversity (e.g., data point color, shape, size, alpha (opacity)). You don't need to fill all four if you used only two factor, set the others to `NULL`.

Once you've set all parameters, click **Assign to R**. These variables wil be saved to R functions for downstream analysis.

**Note:**
* You can modify the beta diversity aesthetics at any time. Update the parameter values, click **Assign to R** again, and then re-run the beta diversity code block (3.16).

In [ ]:
import sys
sys.path.append('/content/Workshop_H2Omics_test/H2Omics_analysis')

from starting_script import analysis_options

analysis_options()

From here, you have two options:

1. Manual execution: Click the play icon next to each code cell to run them one at a time.
2. Automatic execution: To run all cells automatically, place your cursor inside the code block below, then click in the Google Colab menu bar on **Runtime > Run cell and below**.

Once you select **Run cell and below** all code blocks will execute automatically. Depending on the size of your dataset, this may take some time to complete.

## 2.4 Installing and Loading R Packages

In this section, you install R packages required to analyse microbiome data.


In [ ]:
%%R
source("/content/Workshop_H2Omics_test/H2Omics_analysis/install_packages.R")

install_packages()

# 3.0 Starting the analysis

## 3.1 Copy number prediction of 16S rRNA genes

The analysis starts with amplicon sequence variant representative sequences to predict the number of ribosomal RNA gene copies present in the genome of the bacterium an ASV belongs to. The output is a table in which for each ASV (column 'OTU') the predicted gene copy number is included (as well as a prediction confidence probability).


In [ ]:
%%R
source("/content/Workshop_H2Omics_test/H2Omics_analysis/copy_number_prediction.R")

copy_number_prediction()

## 3.2 Creating a Logging System

To improve the tractability and promote reproducibility of your microbiome data analysis, this code block creates a directory "messages" in which a log file is automatically updated (logging_output.txt), which from here onwards keeps track of every step in the analysis.

In [ ]:
%%R
if(!dir.exists(paste0(base_path, projects, "/messages"))){dir.create(paste0(base_path, projects, "/messages"))}
log_file = paste0(base_path, glue("{projects}/messages/logging_output.txt"))

log_message = function(message, log_file) {
  write(paste(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "-", message), log_file, append = TRUE)
}

## 3.3 Set up project structure & merging Metadata

The code block below creates a directory structure for the project, and checks the presence of required input files.

It also merges and reformats metadata, combining QIIME2 metadata with optional extra sample metadata (metadata_extra), and biomass metadata such as qPCR data or FCM data.

In [ ]:
%%R
# create folders and copy files
micromics::create_folders(projects)

# combine qiime2 metadata with experimental sampled metadata
unified_metadata = micromics::unify_metadata(projects)

## 3.4 Creating the phyloseq object

A Phyloseq object holds all key elements together: the feature table, the taxonomy table, phylogenetic tree and unified sample metadata, into a single coherent dataset for streamlined analysis. The code block below creates a phyloseq object from the data you loaded into this notebook.

In [ ]:
%%R
# create a physeq object
physeq = micromics::creating_physeq_object(projects)

## 3.5 Optimizing the taxonomic information

The code block below cleans and filters the taxonomy table of your phyloseq object. It removes unclassified or ambiguous names, and replaces these and missing taxon names at Genus level with placeholders derived from higher taxonomic ranks. For example, if for a given Enterobacteriaceae ASV the Genus name could not be classified with sufficient confidence, the empty cell is filled with "Genus of Enterobacteriaceae" instead of the universal 'unclassified'.

The cleaned taxonomy is then ready to summarize the microbial abundances at higher taxonomic ranks to for example the Genus level. The updated taxonomic information now prevents merging 'unclassified' ASVs from diverse phylogenetic ancestry into a single artifical genus called 'unclassified'.

In [ ]:
%%R
# tax clean
cleaned_physeq = micromics::tax_clean(physeq = physeq, tax_filter = TRUE)

## 3.6 Resolving the phylogenetic tree

This code block performs a similar cleaning step to your phyloseq object, but this time to the phylogenetic tree by resolving all nodes into bifurcations. The original tree is replaced by an updated tree in your phyloseq object.

In [ ]:
%%R
# Resolve polytomous branching of the QIIME2 Fasttree2 phylogeny into a fully bifurcated tree for phylogenetic analyses
resolved_tree_physeq = micromics::resolve_tree(physeq = cleaned_physeq)

## 3.7 Removing contaminant ASVs

In the next code block, contamination is removed from the phyloseq object. This approach requires information from sequenced blank samples (empty DNA samples that thus contain contaminants from the sample processing in the lab) and/or applies a statistical model to designate ASVs as contaminants as predicted by their abundance in samples related to the sample DNA concentration (which is recorded in the sample metadata). In short, contaminant ASVs are more prevalent and likely more abundant in samples with low DNA content. With high DNA conctrations (high biomass samples) contaminants will be outcompeted by true sample-derived bacteria.

This decontamination process also generates figures that illustrate the read counts and prevalence of contaminants across the samples in the Project. For detailed information on the approach, see [Davies et al. 2018](https://microbiomejournal.biomedcentral.com/articles/10.1186/s40168-018-0605-2)

In [ ]:
%%R
# decontam (decon_method = frequency, prevalence or both)
decontam_physeq = micromics::decontam(physeq = resolved_tree_physeq, decon_method = "both", blank = blank)

## 3.8 Removing Mock community ASVs

A mock microbiota sample with known bacteria in known proportions is advised to sequence along with the samples as a positive control. Any mock bacteria that cross-contaminated into the samples are to be removed. The function in the code block removes mock ASVs (and the mock sample) from your phyloseq object.

In this code block, you must specify the mock genera used in your own mock community. An example is shown below, replace the example genus names with the genera you actually used in your mock community. If you did not usse a mock community, you can leave the example genera as the are, and make sure to set mock = FALSE in chapter 2.3. Note that you should use mock taxa that are not native to your study ecosystem.

In [ ]:
%%R
# old mock 2024
mock_genera = c("Massilia", "Serratia", "Vulgatibacter", "Brevibacillus", "Lysinibacillus",
                "Weizmannia", "Streptomyces", "Bacillus", "Peribacillus", "Bordetella",
                "Sphingobium", "Burkholderia-Caballeronia-Paraburkholderia",
                "Genus of Bacilli", "Genus of Bacillaceae", "Genus of Bacillales",
                "Genus of Planococcaceae", "Genus of Burkholderiaceae",
                "Genus of Burkholderiales", "Genus of Yersiniaceae")

# new mock 2025
# mock_genera = c("Abyssicoccus", "Halomonas", "Shewanella", "Alteromonas", "Acetobacter", "Photobacterium", "Caminibacter", "Francisella")

# remove mock and mock features
without_mock_physeq = micromics::remove_mock(physeq = decontam_physeq, mock = TRUE, mock_genera = mock_genera)


## 3.9 Ribosomal gene copy number correction & Biomass normalisation of microbiome

Microbiome analysis is typically performed by sequencing 16S ribosomal RNA fragments, amplified first by PCR. However, the number of copies of 16S rRNA gene copies varies among bacterial species, ranging 1-15 copies per genome. After your phyloseq object has been cleaned from potential contamination, additional steps can be taken to present microbiome data at the 'bacterial cell' level rather than gene copy level.

In the code block below, each ASV sequence count is divided by the ASV's predicted ribosomal gene copy nummber (predicted by the copy_number_prediction() function performed earlier). This copy number correction reduces an overestimation of the abundance of bacteria that have more rRNA gene copies than average, and the opposite is true for bacteria with fewer than average ribosomal RNA gene copies.

Next, you will combine the microbiome data with your measurement of biomass in the samples. You will account for biomass differences among your samples before visualization or statistical hypothesis testing. This biomass normalization helps obtaining more meaningful interpretation of your microbiome data. For example, considering identical relative contributions of all bacteria, comparing a sample with  100  cells/mL  with a sample with  100000  cells/mL  with will look very different after the factor 1000 difference is integrated before visualizing results. The normalized compositions will more accurately reflect the true microbial composition of the sampled ecosystem. In the data import stage earlier on, you have selected which type of biomass data you will use, either flow cytometry or qPCR data.

Going forward, the resulting unit of analysis will change from sequence count to (approximated) cell count (per unit sample, such as per liter or per gram sample), which will be more meaningful in a public health context and for infection risk assessment. It is optional to apply this correction. Choose `TRUE` or `FALSE`, and confirm your choice.

In [ ]:
%%R
# copy number correction and biomasss normalisation
normalised_asv_physeq = micromics::normalise_data(physeq = without_mock_physeq, norm_method = norm_method, copy_correction = TRUE)

## 3.10 Rarefaction

After copy correction and biomass normalisation, rarefaction is performed. Rarefying is the process random subsampling an equal number of units from every sample. Comparing microbiomes after rarefaction makes sure to compare microbiome samples more fairly and interpret differences in with minimal bias. Rarefaction is applied on the normalized phyloseq object created in the previous step. How much to subsample for rarefying your samples is determined by the biomass in each sample (how well is my sample representing the ecosystem?) and the number of sequences obtained from each sample (how elaborate is my measurement?). These are combined to define the minimum sampling depth across the dataset. For documentation of this approach, see [Vandeputte et al., 2017](https://www.nature.com/articles/nature24460).

In [ ]:
%%R
# rarefied data
rarefied_asv_physeq = micromics::rarefying(physeq = normalised_asv_physeq, norm_method = norm_method, iteration = 10)

## 3.11 Grouping data by taxonomy level

The code below facilitates aggregaing your microbiome data at each higher taxonomic level (i.e., Phylum, Class, Order, Family, and Genus). Depending on the specified normalization method, the function processes and saves both only copy number corrected (cnc) data, as well as  cnc+biomass normalized data (using flow cytometry or qPCR).

In [ ]:
%%R
rarefied_tax_physeq = micromics::group_tax(physeq = rarefied_asv_physeq, norm_method = norm_method, copy_correction = TRUE)

# converting phyloseq object to a tibble
rarefied_tax_psmelt = micromics::psdata_to_tibble(physeq = rarefied_tax_physeq, norm_method = norm_method)

The following code blocks create different types of plots. These plots are all saved and can be viewed in Google Drive.

## 3.12 Create a Barplot

The code block below facilitates generating barplots of microbiome data at the genus level (limited to the genus level, for simplicity). It supports visualizing relative abudances (ranging 0-100%) and normalized cell concentrations. You can now organize your barplot using experimental factors as supplied in the sample metadata. The resulting plots can be saved as PDF files, and the underlying data can be exported as MS Excel readable .CSV and R readable .RDS files.

In [ ]:
%%R -w 10 -h 8 -u in
# Barplot
created_barplot = micromics::barplot(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method,
                                     sample_matrix = "liquid", group_by_factor = NULL)

## 3.13 Creating a Heatmap

The barplots help assess differences among samples, but comparing bacteria is less intuitive. In the code block below, a heatmap is created visualizing the relative abundance data at the genus level. Similar to the barplots, bacteria that are below a defined abundance threshold are grouped into a category "other".


In [ ]:
%%R -w 10 -h 8 -u in
# heatmap
heatmap_plot = micromics::heatmap(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method)

## 3.14 Alpha diversity: the bacterial richness and diversity within samples

The richness and diversity of bacteria detected within each sample can be compared at the ASV level or at different taxonomic levels. Alpha diversity metrics can provide distinct insights into this diversity, such as the observed number of taxa, the estimated total number of taxa (Chao1), or taking into account the degree of dominance of taxa (Shannon and Simpson indices).

In [ ]:
%%R -w 12 -h 7 -u in
options(warn = -1)
# Alpha diversity

# ASV level
alpha_div_plots = micromics::alpha_diversity(physeq = rarefied_asv_physeq, taxrank = "asv", norm_method = norm_method)

# Genus level
alpha_div_plots = micromics::alpha_diversity(physeq = rarefied_tax_physeq, norm_method = norm_method)

## 3.15 Beta diversity: pairwise comparing microbiome samples

Beta diversity describes the degree of clustering of samples based on microbiome composition. It helps summarize compositional differentiation among groups of samples. Beta diversity is calculated by pairwise comparison of microbiome compositions. Several pairwise differences can be calculated with distinct ecological interpretations. These options depend on taking as input only presence/absence (0/1) of bacteria (Jaccard index), the counts or relative abundances of detected taxa (Bray-Curtis index), or either of these options but also taking the relatedness among the taxa into account (UniFrac metric). Visualization of beta diversity is typically a principal coordinates analysis (PCoA) plot where x and y axes depict most of variation of the sample clustering. Beta diversity can be calculated at all taxonomic levels, and using relative or normalised counts data. For the latter, another distance metric is suitable (Manhattan distance).

In [ ]:
%%R -w 8 -h 6 -u in
options(warn = -1)
# Beta diversity

# ASV level
beta_div_plots = micromics::beta_diversity(physeq = rarefied_asv_physeq, taxrank = "asv", norm_method = norm_method,
                                           ordination_method = "PCoA", color_factor = color_factor, color_continuous = FALSE,
                                           shape_factor = shape_factor, size_factor = size_factor, alpha_factor = alpha_factor)

# Genus level
beta_div_plots = micromics::beta_diversity(physeq = rarefied_tax_physeq, norm_method = norm_method,
                                           ordination_method = "PCoA", color_factor = color_factor, color_continuous = FALSE,
                                           shape_factor = shape_factor, size_factor = size_factor, alpha_factor = alpha_factor)

## 3.16 Exporting results and figures

The final code block of this notebook helps you to export results, output data and figures by creating a dedicated export folder within the project directory you created at the start of the notebook. It organises the export folder into subdirectories for figures, CSV files, and RDS files, and creates copies of the most relevant files from their original locations on your Google Drive.

In [ ]:
%%R
# Export data
micromics::export_data()

# License
This package is licensed under the MIT License with additional terms restricting its use to non-commercial research only. See the [LICENSE](https://github.com/stijnteunissen/micromics/blob/master/LICENSE.md) file for details.